In [1]:
import numpy as np 
import pandas as pd

In [2]:
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

In [3]:
movies.dtypes

budget                    int64
genres                   object
homepage                 object
id                        int64
keywords                 object
original_language        object
original_title           object
overview                 object
popularity              float64
production_companies     object
production_countries     object
release_date             object
revenue                   int64
runtime                 float64
spoken_languages         object
status                   object
tagline                  object
title                    object
vote_average            float64
vote_count                int64
dtype: object

In [4]:
credits.dtypes

movie_id     int64
title       object
cast        object
crew        object
dtype: object

In [5]:
movies = movies.merge(credits,on="title")
movies.dtypes

budget                    int64
genres                   object
homepage                 object
id                        int64
keywords                 object
original_language        object
original_title           object
overview                 object
popularity              float64
production_companies     object
production_countries     object
release_date             object
revenue                   int64
runtime                 float64
spoken_languages         object
status                   object
tagline                  object
title                    object
vote_average            float64
vote_count                int64
movie_id                  int64
cast                     object
crew                     object
dtype: object

In [6]:
movies = movies[['movie_id','title','genres','keywords','overview','cast','crew','release_date','runtime','tagline','budget','revenue']]

In [7]:
movies.isnull().sum()

movie_id          0
title             0
genres            0
keywords          0
overview          3
cast              0
crew              0
release_date      1
runtime           2
tagline         844
budget            0
revenue           0
dtype: int64

In [8]:
movies['overview'].fillna('',inplace=True)
movies.fillna("Unknown",inplace=True)

In [9]:
movies.isnull().sum()

movie_id        0
title           0
genres          0
keywords        0
overview        0
cast            0
crew            0
release_date    0
runtime         0
tagline         0
budget          0
revenue         0
dtype: int64

In [10]:
movies.duplicated().sum()

0

In [11]:
import ast
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [12]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [13]:
def convert5(obj):
    c = 0
    L = []
    for i in ast.literal_eval(obj):
        if c != 5:
            L.append(i['name'])
            c=+1
        else:
            break
    return L

In [14]:
movies['cast'] = movies['cast'].apply(convert5)

In [15]:
def fetch_dir(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L

In [16]:
movies['crew'] = movies['crew'].apply(fetch_dir)

In [17]:
o = movies['overview'].apply(lambda x:x.split())
g = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
k = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
c = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
d = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

movies["tag"] = o+g+k+c+d

In [18]:
movies['tag'] = movies['tag'].apply(lambda x:" ".join(x))

In [19]:
movies['tag'] = movies['tag'].apply(lambda x:x.lower())

In [20]:
new = movies[['movie_id','title','tag']]

In [21]:
import nltk
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [22]:
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [23]:
new['tag'] = new['tag'].apply(stem)
new

/tmp/ipykernel_2629/3914407930.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new['tag'] = new['tag'].apply(stem)


,movie_id,title,tag
0,19995,Avatar,"in the 22nd century, a parapleg marin is dispa..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha c..."
2,206647,Spectre,a cryptic messag from bond’ past send him on a...
3,49026,The Dark Knight Rises,follow the death of district attorney harvey d...
4,49529,John Carter,"john carter is a war-weary, former militari ca..."
...,...,...,...
4804,9367,El Mariachi,el mariachi just want to play hi guitar and ca...
4805,72766,Newlyweds,a newlyw couple' honeymoon is upend by the arr...
4806,231617,"Signed, Sealed, Delivered","""signed, sealed, delivered"" introduc a dedic q..."
4807,126186,Shanghai Calling,when ambiti new york attorney sam is sent to s...


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000,stop_words='english') 

In [25]:
vectors = cv.fit_transform(new['tag']).toarray()
vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [26]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)

In [27]:
import pickle
pickle.dump(movies,open('movie_list.pkl','wb'))
pickle.dump(similarity,open('similarity.pkl','wb'))

In [28]:
pip install rapidfuzz

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


In [29]:
from rapidfuzz import process

def recommend(movie):
    movie_titles = movies['title'].tolist()

    match, score, _ = process.extractOne(movie, movie_titles)

    if score < 50:
        print("Movie not found")
        return

    print(f"\nRecommendations for '{match}':\n")

    index = movies[movies['title'] == match].index[0]

    distances = sorted(
        list(enumerate(similarity[index])),
        key=lambda x: x[1],
        reverse=True
    )

    for i in distances[1:6]:
        print("•", new.iloc[i[0]]['title'])

In [31]:
recommend("avengers")


Recommendations for 'The Avengers':

• Avengers: Age of Ultron
• Iron Man 3
• Iron Man
• Captain America: Civil War
• Captain America: The First Avenger
